# Amazon Product Funnel — Final Cleaning Pass

Adds the `is_discounted` flag and casts binary columns to nullable integer types
for clean ingestion into BigQuery.

- **Input:** `cleanest_amazon_products.csv`
- **Output:** `cleanest_amazon_products_v4.csv`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\user\Desktop\Data_Analytics_Project2\Data files(CSVs)\May 9(cleanest_version_reproducible_for_Github)\cleanest_amazon_products.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   asin                224 non-null    object 
 1   productDescription  224 non-null    object 
 2   category            224 non-null    object 
 3   price               218 non-null    float64
 4   retailPrice         224 non-null    float64
 5   productRating       223 non-null    float64
 6   countReview         224 non-null    int64  
 7   salesVolume_clean   218 non-null    float64
 8   price_tier          218 non-null    object 
 9   has_reviews         224 non-null    int64  
 10  review_density      223 non-null    float64
 11  prime               224 non-null    bool   
dtypes: bool(1), float64(5), int64(2), object(4)
memory usage: 19.6+ KB


## 1. Discount Flag

A `retailPrice` of `0.0` indicates no original price was available — replaced with `NaN`
before comparison. Records missing either value receive `NaN` rather than a false `0`.

In [2]:
df["retailPrice"] = df["retailPrice"].replace(0, np.nan)

df["is_discounted"] = np.where(
    df["retailPrice"].notna() & df["price"].notna(),
    (df["price"] < df["retailPrice"]).astype(int),
    np.nan
)

print(f"Products with retail price data: {df['is_discounted'].notna().sum()}")
print(f"Of those, discounted: {df['is_discounted'].sum():.0f}")
df[["price", "retailPrice", "is_discounted"]].sample(10, random_state=42)

Products with retail price data: 78
Of those, discounted: 78


,price,retailPrice,is_discounted
9,16.84,19.99,1.0
84,24.99,NaN,NaN
117,30.42,32.07,1.0
144,18.48,NaN,NaN
221,149.99,159.99,1.0
113,21.49,NaN,NaN
68,59.99,NaN,NaN
104,6.98,NaN,NaN
177,NaN,NaN,NaN
185,18.48,NaN,NaN


## 2. Nullable Integer Types

Pandas `Int64` (capital I) preserves `<NA>` instead of upcasting to `float64`,
keeping binary flags semantically correct for BigQuery ingestion.

In [3]:
df["has_reviews"] = df["has_reviews"].astype("Int64")
df["is_discounted"] = df["is_discounted"].astype("Int64")

print(df[["has_reviews", "is_discounted"]].dtypes)
print("\nis_discounted value counts (including NA):")
print(df["is_discounted"].value_counts(dropna=False))

has_reviews      Int64
is_discounted    Int64
dtype: object

is_discounted value counts (including NA):
is_discounted
<NA>    146
1        78
Name: count, dtype: Int64


## 3. Export Final Dataset

In [4]:
df.to_csv("cleanest_amazon_products_v4.csv", index=False)
print("Final dataset exported: cleanest_amazon_products_v4.csv")

Final dataset exported: cleanest_amazon_products_v4.csv
